# 08 — Module 4: LLM-agent validation

Do XAI explanations actually help a modern language model make better
fraud-vs-legitimate decisions? Three frontier agents are queried under
three information conditions on 100 deliberately ambiguous transactions
from each dataset, for the best and worst BRAS-scoring XAI
configuration.

**Design choices (applied here):**

- **P1 Multi-configuration.** For each dataset we pick the highest-BRAS
  and lowest-BRAS ML configuration from Module 3, so the effect of
  explanation quality on decisions can be isolated.
- **P2 Ambiguous transactions.** Only instances with model-predicted
  `P(fraud) ∈ [0.3, 0.7]` are considered (progressive fallback to wider
  windows if too few). This removes the ceiling effect and forces the
  agent to rely on the explanation.
- **P3 Feature grouping.** Elliptic's anonymized features are mapped to
  semantic groups (structure, amount, temporal, neighborhood) so the
  agent has something interpretable to reason about.
- **P4 Non-prescriptive C1.** C1 shows raw features only (no model
  probability, no XAI).
- **P5 Non-prescriptive C3.** C3 shows the nine Module-1/2/3 scores but
  does not tell the agent how to weight them.

**Agents (via OpenRouter):**

- Claude Opus 4.7 (`anthropic/claude-opus-4.7`)
- Gemini 3.1 Pro preview (`google/gemini-3.1-pro-preview`)
- GPT 5.4 (`openai/gpt-5.4`)

**Prerequisite:** notebooks 02a, 02b, 03a, 03b, 04, 05, 06 have been
executed and a valid `OPENROUTER_API_KEY` is set in `.env`.


In [ ]:
import os
import time
from datetime import datetime
from itertools import combinations

import joblib
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score
import warnings
warnings.filterwarnings('ignore')

from xai_blockchain_framework.config import (
    PATHS, CONFIG, DEFAULT_MODELS, AGENT_DISPLAY_NAMES, set_seed,
)
from xai_blockchain_framework.llm import (
    OpenRouterClient,
    build_prompts,
    parse_response,
    SYSTEM_PROMPT,
)
from xai_blockchain_framework.metrics import (
    decision_accuracy,
    expected_calibration_error,
    explanation_utilization,
    cohen_kappa_pair,
    mean_inter_agent_kappa,
)

set_seed(42)

RESULTS_PATH = str(PATHS.results_dir)
FIGURES_PATH = str(PATHS.figures_dir)
MODELS_PATH = str(PATHS.models_dir)
PROCESSED_PATH = str(PATHS.data_processed)
for p in [RESULTS_PATH, FIGURES_PATH, PROCESSED_PATH]:
    os.makedirs(p, exist_ok=True)

plt.style.use('seaborn-v0_8-whitegrid')

# Experiment parameters (match the research run)
N_TX = 100
TOP_K = 5
N_CAL_BINS = 10
TEMPERATURE = 0.0        # deterministic decoding
MAX_TOKENS = 800
SLEEP = 1.0              # seconds between calls
AMBIG_LOW = 0.3
AMBIG_HIGH = 0.7

# The three agents, keyed by short identifier
AGENTS_SHORT = ['opus', 'gemini', 'gpt']
AGENT_NAMES = [AGENT_DISPLAY_NAMES[s] for s in AGENTS_SHORT]

print("=" * 70)
print("MODULE 4  LLM-AGENT VALIDATION")
print("=" * 70)
print(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"N_transactions={N_TX}  TOP_K={TOP_K}  temperature={TEMPERATURE}")
print(f"Ambiguity window: P(fraud) in [{AMBIG_LOW}, {AMBIG_HIGH}]")
for short, name in zip(AGENTS_SHORT, AGENT_NAMES):
    print(f"  Agent: {name}  ({DEFAULT_MODELS[short]})")

ETHEREUM_FEATURES = [
    'Avg min between sent tnx', 'Avg min between received tnx',
    'Time Diff between first and last', 'Sent tnx', 'Received Tnx',
    'Number of Created Contracts', 'Unique Received From Addresses',
    'Unique Sent To Addresses', 'Min Value Received', 'Max Value Received',
    'Avg Value Received', 'Min Val Sent', 'Max Val Sent', 'Avg Val Sent',
    'Min Value Sent To Contract', 'Max Value Sent To Contract',
    'Avg Value Sent To Contract', 'Total Transactions (incl tnx to create contract)',
    'Total Ether Sent', 'Total Ether Received', 'Total Ether Sent Contracts',
    'Total ERC20 Tnx', 'ERC20 Total Ether Received', 'ERC20 Total Ether Sent',
    'ERC20 Total Ether Sent Contract', 'ERC20 Uniq Sent Addr',
    'ERC20 Uniq Rec Addr', 'ERC20 Uniq Sent Addr.1',
    'ERC20 Uniq Rec Contract Addr', 'ERC20 Min Val Rec',
    'ERC20 Max Val Rec', 'ERC20 Avg Val Rec', 'ERC20 Min Val Sent',
    'ERC20 Max Val Sent', 'ERC20 Avg Val Sent', 'ERC20 Uniq Sent Token Name',
    'ERC20 Uniq Rec Token Name', 'ERC20 Most Sent Token Type',
    'ERC20 Most Rec Token Type', 'Total Ether Balance',
    'ERC20 Most Sent Token Type_ratio', 'ERC20 Most Rec Token Type_ratio',
    'Sent_Received_ratio', 'Ether_ERC20_ratio', 'Contracts_ratio',
]


## 1. Load models, splits, attributions, and Module 1-3 scores


In [ ]:
print("\n" + "=" * 70)
print("1. Loading")
print("=" * 70)

ell = np.load(os.path.join(PROCESSED_PATH, 'elliptic_splits.npz'))
eth = np.load(os.path.join(PROCESSED_PATH, 'ethereum_splits.npz'))
X_test_ell, y_test_ell = ell['X_test'], ell['y_test']
X_test_eth, y_test_eth = eth['X_test'], eth['y_test']
eval_idx_ell = np.load(os.path.join(RESULTS_PATH, 'xai_eval_indices_elliptic.npy'))
eval_idx_eth = np.load(os.path.join(RESULTS_PATH, 'xai_eval_indices_ethereum.npy'))

rf_ell  = joblib.load(os.path.join(MODELS_PATH, 'elliptic_rf.joblib'))
lgb_ell = joblib.load(os.path.join(MODELS_PATH, 'elliptic_lgb.joblib'))
rf_eth  = joblib.load(os.path.join(MODELS_PATH, 'ethereum_rf.joblib'))
lgb_eth = joblib.load(os.path.join(MODELS_PATH, 'ethereum_lgb.joblib'))

shap_rf_ell  = np.load(os.path.join(RESULTS_PATH, 'shap_values_rf_elliptic.npy'))
shap_lgb_ell = np.load(os.path.join(RESULTS_PATH, 'shap_values_lgb_elliptic.npy'))
shap_rf_eth  = np.load(os.path.join(RESULTS_PATH, 'shap_values_rf_ethereum.npy'))
shap_lgb_eth = np.load(os.path.join(RESULTS_PATH, 'shap_values_lgb_ethereum.npy'))
lime_rf_ell  = np.load(os.path.join(RESULTS_PATH, 'lime_attrs_rf_elliptic.npy'))
lime_lgb_ell = np.load(os.path.join(RESULTS_PATH, 'lime_attrs_lgb_elliptic.npy'))
lime_rf_eth  = np.load(os.path.join(RESULTS_PATH, 'lime_attrs_rf_ethereum.npy'))
lime_lgb_eth = np.load(os.path.join(RESULTS_PATH, 'lime_attrs_lgb_ethereum.npy'))

m1 = pd.read_csv(os.path.join(RESULTS_PATH, 'module1_fidelity_results.csv'))
m2 = pd.read_csv(os.path.join(RESULTS_PATH, 'module2_stability_results.csv'))
m3 = pd.read_csv(os.path.join(RESULTS_PATH, 'module3_bras_results.csv'))
if 'k' in m1.columns:
    m1 = m1[m1['k'] == TOP_K]

print(f"  M1 rows: {len(m1)}  M2 rows: {len(m2)}  M3 rows: {len(m3)}")


## 2. OpenRouter client + smoke-test all three models


In [ ]:
print("\n" + "=" * 70)
print("2. OpenRouter client")
print("=" * 70)

client = OpenRouterClient()

active_agents: list[str] = []
for short in AGENTS_SHORT:
    model_id = DEFAULT_MODELS[short]
    display = AGENT_DISPLAY_NAMES[short]
    try:
        client.call(model=model_id, system="OK", user="OK", max_tokens=16, temperature=TEMPERATURE)
        active_agents.append(short)
        print(f"  {display} ({model_id}) -> OK")
    except Exception as e:
        print(f"  {display} ({model_id}) -> FAILED  ({e})")

if not active_agents:
    raise SystemExit("No agent is reachable. Verify OPENROUTER_API_KEY and model IDs.")

AGENT_DISPLAY = [AGENT_DISPLAY_NAMES[s] for s in active_agents]
print(f"\n{len(active_agents)}/3 agents active: {AGENT_DISPLAY}")


## 3. Helpers: ambiguous transaction selection + quality-score lookup

These helpers live in the notebook because they wire together data
artifacts that only this notebook has: the per-configuration
attributions, the saved M1/M2/M3 CSVs, and the model prediction
probabilities.


In [ ]:
ATTRIBUTIONS = {
    ('RF',  'SHAP', 'Elliptic'): (rf_ell,  shap_rf_ell),
    ('LGB', 'SHAP', 'Elliptic'): (lgb_ell, shap_lgb_ell),
    ('RF',  'SHAP', 'Ethereum'): (rf_eth,  shap_rf_eth),
    ('LGB', 'SHAP', 'Ethereum'): (lgb_eth, shap_lgb_eth),
    ('RF',  'LIME', 'Elliptic'): (rf_ell,  lime_rf_ell),
    ('LGB', 'LIME', 'Elliptic'): (lgb_ell, lime_lgb_ell),
    ('RF',  'LIME', 'Ethereum'): (rf_eth,  lime_rf_eth),
    ('LGB', 'LIME', 'Ethereum'): (lgb_eth, lime_lgb_eth),
}


def get_quality_scores(model_name: str, xai_name: str, dataset: str) -> dict[str, float]:
    '''Join M1, M2, M3 rows that match (Model, XAI, Dataset) into one dict.'''
    scores: dict[str, float] = {}
    sources = [
        (m1, ['Comprehensiveness', 'Sufficiency', 'Infidelity']),
        (m2, ['Lipschitz_norm', 'Kendall_tau', 'CoV_Bootstrap', 'Identity']),
        (m3, ['RAS', 'DVR', 'BRAS']),
    ]
    for df, cols in sources:
        if df.empty:
            continue
        mask = pd.Series(True, index=df.index)
        if 'Model' in df.columns:   mask &= df['Model']   == model_name
        if 'XAI'   in df.columns:   mask &= df['XAI']     == xai_name
        if 'Dataset' in df.columns: mask &= df['Dataset'] == dataset
        match = df[mask]
        if len(match) == 0:
            continue
        row = match.iloc[0]
        for c in cols:
            if c in row.index and pd.notna(row[c]):
                scores[c] = float(row[c])
    return scores


def select_ambiguous_transactions(y, eval_idx, model, X_test, n=N_TX, seed=42):
    '''Sample up to n balanced ambiguous indices (progressive fallback).'''
    rng = np.random.RandomState(seed)
    probas = model.predict_proba(X_test[eval_idx])[:, 1]

    windows = [(AMBIG_LOW, AMBIG_HIGH), (0.2, 0.8), (0.1, 0.9)]
    ambig_idx = np.array([], dtype=eval_idx.dtype)
    for lo, hi in windows:
        mask = (probas >= lo) & (probas <= hi)
        if mask.sum() >= n:
            ambig_idx = eval_idx[mask]
            break
    if len(ambig_idx) < n:
        # Last resort: sort by distance from 0.5
        order = np.argsort(np.abs(probas - 0.5))
        ambig_idx = eval_idx[order[:n]]

    fraud = ambig_idx[y[ambig_idx] == 1]
    legit = ambig_idx[y[ambig_idx] == 0]
    nf = min(n // 2, len(fraud))
    nl = min(n - nf, len(legit))
    if nf + nl < n:
        nf = min(n - nl, len(fraud))
    selected = np.concatenate([
        rng.choice(fraud, nf, replace=False) if nf > 0 else np.array([], dtype=ambig_idx.dtype),
        rng.choice(legit, nl, replace=False) if nl > 0 else np.array([], dtype=ambig_idx.dtype),
    ])
    rng.shuffle(selected)
    sel_probas = model.predict_proba(X_test[selected])[:, 1]
    print(f"    selected {len(selected)} tx  "
          f"(fraud={int((y[selected] == 1).sum())}  legit={int((y[selected] == 0).sum())})  "
          f"P(fraud) in [{sel_probas.min():.3f}, {sel_probas.max():.3f}]")
    return selected


def select_bras_configs(m3_df, dataset):
    '''Pick the highest-BRAS and lowest-BRAS ML config for a dataset.'''
    sub = m3_df[(m3_df['Dataset'] == dataset) & (m3_df['Type'] == 'ML')]
    if sub.empty:
        sub = m3_df[m3_df['Dataset'] == dataset]
    if sub.empty or 'BRAS' not in sub.columns:
        return [('LGB', 'SHAP', 'baseline')]
    best = sub.loc[sub['BRAS'].idxmax()]
    worst = sub.loc[sub['BRAS'].idxmin()]
    out = [(best['Model'], best['XAI'], f"high_BRAS={best['BRAS']:.3f}")]
    if worst['BRAS'] != best['BRAS']:
        out.append((worst['Model'], worst['XAI'], f"low_BRAS={worst['BRAS']:.3f}"))
    return out


## 4. Build the configuration list


In [ ]:
print("\n" + "=" * 70)
print("3. Selecting configurations")
print("=" * 70)

configs: list[dict] = []
for ds_name, X_test, y_test, eval_idx, feature_names in [
    ('Elliptic', X_test_ell, y_test_ell, eval_idx_ell, None),
    ('Ethereum', X_test_eth, y_test_eth, eval_idx_eth, ETHEREUM_FEATURES),
]:
    print(f"\n--- {ds_name} ---")
    for model_name, xai_name, bras_label in select_bras_configs(m3, ds_name):
        model, attrs = ATTRIBUTIONS.get((model_name, xai_name, ds_name), (None, None))
        if model is None:
            print(f"  {model_name}-{xai_name}: attribution not found, skipping")
            continue
        quality = get_quality_scores(model_name, xai_name, ds_name)
        print(f"  Config: {model_name}-{xai_name}  ({bras_label})")
        tx_ids = select_ambiguous_transactions(y_test, eval_idx, model, X_test, n=N_TX)
        configs.append({
            'dataset': ds_name, 'model_name': model_name, 'xai_name': xai_name,
            'bras_label': bras_label,
            'model': model, 'attrs': attrs,
            'X_test': X_test, 'y_test': y_test,
            'tx_ids': tx_ids, 'eval_idx': eval_idx,
            'feature_names': feature_names, 'quality': quality,
        })
print(f"\n{len(configs)} configuration(s) to evaluate")


## 5. Main evaluation loop

Each transaction × each of C1/C2/C3 × each agent = one API call. Total
count scales as `len(configs) * N_TX * 3 * len(active_agents)`.


In [ ]:
print("\n" + "=" * 70)
print("4. Evaluation")
print("=" * 70)

responses: list[dict] = []
for cfg in configs:
    ds, mn, xn, bl = cfg['dataset'], cfg['model_name'], cfg['xai_name'], cfg['bras_label']
    model, attrs, X_test, y_test = cfg['model'], cfg['attrs'], cfg['X_test'], cfg['y_test']
    eval_idx, tx_ids = cfg['eval_idx'], cfg['tx_ids']
    feature_names = cfg['feature_names']
    quality = cfg['quality']

    print(f"\n=== {ds}  {mn}-{xn}  ({bl}) ===")
    print(f"    {len(tx_ids)} tx x 3 conditions x {len(active_agents)} agents")

    for t, ti in enumerate(tx_ids):
        positions = np.where(eval_idx == ti)[0]
        if len(positions) == 0 or positions[0] >= len(attrs):
            continue
        ai = int(positions[0])
        gt = 'fraud' if int(y_test[ti]) == 1 else 'legitimate'
        p_fraud = float(model.predict_proba(X_test[ti].reshape(1, -1))[0, 1])

        prompts = build_prompts(
            dataset=ds,
            x=X_test[ti],
            model_name=mn,
            model_fraud_probability=p_fraud,
            attributions=attrs[ai],
            feature_names=feature_names,
            quality_scores=quality,
            top_k=TOP_K,
        )

        for cond in ('C1', 'C2', 'C3'):
            for short in active_agents:
                display_name = AGENT_DISPLAY_NAMES[short]
                model_id = DEFAULT_MODELS[short]
                try:
                    resp = client.call(
                        model=model_id,
                        system=SYSTEM_PROMPT,
                        user=prompts[cond],
                        temperature=TEMPERATURE,
                        max_tokens=MAX_TOKENS,
                        response_format={'type': 'json_object'},
                    )
                    parsed = parse_response(resp.content)
                except Exception as e:
                    print(f"    {display_name} {cond}: FAILED ({e})")
                    parsed = parse_response(None)

                eu = (
                    explanation_utilization(
                        parsed.reasoning, attrs[ai],
                        feature_names or [f"f{i}" for i in range(len(attrs[ai]))],
                        k=TOP_K,
                    )
                    if cond in ('C2', 'C3') else None
                )
                responses.append({
                    'Dataset': ds, 'Model': mn, 'XAI': xn,
                    'BRAS_label': bl, 'Condition': cond,
                    'Agent': display_name, 'TX_idx': int(ti),
                    'Ground_Truth': gt, 'P_fraud': p_fraud,
                    'Decision': parsed.decision, 'Confidence': parsed.confidence,
                    'EU': eu, 'Reasoning': parsed.reasoning,
                    'Explanation': parsed.explanation,
                })
                if SLEEP > 0:
                    time.sleep(SLEEP)
        if (t + 1) % 10 == 0:
            print(f"    {t + 1}/{len(tx_ids)} tx")

resp_df = pd.DataFrame(responses)
resp_df.to_csv(os.path.join(RESULTS_PATH, 'module4v3_llm_raw_responses.csv'), index=False)
print(f"\n{len(resp_df)} responses collected")
if len(resp_df) == 0 or resp_df['Decision'].notna().sum() == 0:
    raise SystemExit("No valid responses: verify OPENROUTER_API_KEY and model availability.")


## 6. Metrics per (Dataset, Config, Condition, Agent)


In [ ]:
print("\n" + "=" * 70)
print("5. Metrics")
print("=" * 70)

vdf = resp_df[resp_df['Decision'].notna()].copy()
vdf['Dec_bin'] = (vdf['Decision'] == 'fraud').astype(int)
vdf['GT_bin']  = (vdf['Ground_Truth'] == 'fraud').astype(int)
vdf['Correct'] = (vdf['Dec_bin'] == vdf['GT_bin']).astype(int)
print(f"Valid: {len(vdf)}/{len(resp_df)}")

rows: list[dict] = []
for ds in vdf['Dataset'].unique():
    for bl in vdf[vdf['Dataset'] == ds]['BRAS_label'].unique():
        for cond in ['C1', 'C2', 'C3']:
            cond_df = vdf[(vdf['Dataset'] == ds) & (vdf['BRAS_label'] == bl) & (vdf['Condition'] == cond)]
            if cond_df.empty:
                continue

            # Inter-agent kappa (aligned on the common TX indices).
            per_agent_decisions = {}
            for ag in AGENT_DISPLAY:
                sub = cond_df[cond_df['Agent'] == ag].set_index('TX_idx')['Decision']
                if not sub.empty:
                    per_agent_decisions[ag] = sub
            kappa = float('nan')
            if len(per_agent_decisions) >= 2:
                pair_kappas = []
                for a, b in combinations(per_agent_decisions.keys(), 2):
                    common = per_agent_decisions[a].index.intersection(per_agent_decisions[b].index)
                    if len(common) >= 2:
                        pair_kappas.append(cohen_kappa_pair(
                            per_agent_decisions[a].loc[common],
                            per_agent_decisions[b].loc[common],
                        ))
                pair_kappas = [k for k in pair_kappas if not np.isnan(k)]
                kappa = float(np.mean(pair_kappas)) if pair_kappas else float('nan')

            for ag in AGENT_DISPLAY:
                sub = cond_df[cond_df['Agent'] == ag]
                if sub.empty:
                    continue
                da = decision_accuracy(sub['Decision'], sub['GT_bin'])
                ece = expected_calibration_error(
                    sub['Confidence'].astype(float), sub['Correct'].astype(int), n_bins=N_CAL_BINS,
                )
                eu_vals = sub['EU'].dropna().astype(float)
                eu_mean = float(eu_vals.mean()) if not eu_vals.empty else None
                rows.append({
                    'Dataset': ds, 'BRAS_label': bl, 'Condition': cond, 'Agent': ag,
                    'DA': float(da), 'ECE': float(ece),
                    'EU': eu_mean, 'Kappa': kappa, 'N': len(sub),
                })
met = pd.DataFrame(rows)
print(met.to_string(index=False))


## 7. Deltas and high-BRAS vs low-BRAS comparison


In [ ]:
deltas = []
for ds in met['Dataset'].unique():
    for bl in met[met['Dataset'] == ds]['BRAS_label'].unique():
        for ag in AGENT_DISPLAY:
            ad = met[(met['Dataset'] == ds) & (met['BRAS_label'] == bl) & (met['Agent'] == ag)]
            c1r = ad[ad['Condition'] == 'C1']
            c2r = ad[ad['Condition'] == 'C2']
            c3r = ad[ad['Condition'] == 'C3']
            if not c1r.empty and not c2r.empty:
                deltas.append({'Dataset': ds, 'BRAS_label': bl, 'Agent': ag, 'Delta': 'C1->C2',
                               'dDA': c2r.iloc[0]['DA'] - c1r.iloc[0]['DA'],
                               'dECE': c2r.iloc[0]['ECE'] - c1r.iloc[0]['ECE']})
            if not c2r.empty and not c3r.empty:
                deltas.append({'Dataset': ds, 'BRAS_label': bl, 'Agent': ag, 'Delta': 'C2->C3',
                               'dDA': c3r.iloc[0]['DA'] - c2r.iloc[0]['DA'],
                               'dECE': c3r.iloc[0]['ECE'] - c2r.iloc[0]['ECE']})
delt_df = pd.DataFrame(deltas)
if not delt_df.empty:
    print(delt_df.to_string(index=False))

print("\n--- High-BRAS vs low-BRAS (mean DA across agents) ---")
for ds in met['Dataset'].unique():
    for cond in ['C1', 'C2', 'C3']:
        high = met[(met['Dataset'] == ds)
                   & met['BRAS_label'].str.contains('high')
                   & (met['Condition'] == cond)]['DA'].mean()
        low = met[(met['Dataset'] == ds)
                  & met['BRAS_label'].str.contains('low')
                  & (met['Condition'] == cond)]['DA'].mean()
        if not (np.isnan(high) or np.isnan(low)):
            print(f"  {ds} {cond}: high={high:.3f}  low={low:.3f}  delta={high - low:+.3f}")


## 8. Visualizations


In [ ]:
print("\n" + "=" * 70)
print("6. Visualizations")
print("=" * 70)

COLORS_COND = {'C1': '#3498db', 'C2': '#2ecc71', 'C3': '#e74c3c'}
COLORS_AGENT = {
    'Claude Opus 4.7': '#d97706',
    'Gemini 3.1 Pro':  '#4285f4',
    'GPT 5.4':         '#10a37f',
}

def _plot_grouped_bar(df, value_col, title, out_name, ylim=(0, 1.05)):
    groups = list(df.groupby(['Dataset', 'BRAS_label']))
    n = len(groups)
    fig, axes = plt.subplots(1, n, figsize=(7 * n, 7), squeeze=False)
    for idx, ((ds, bl), grp) in enumerate(groups):
        ax = axes[0, idx]
        agents = list(dict.fromkeys(grp['Agent']))
        x = np.arange(len(agents))
        w = 0.25
        for j, cond in enumerate(['C1', 'C2', 'C3']):
            vals = []
            for ag in agents:
                sub = grp[(grp['Agent'] == ag) & (grp['Condition'] == cond)][value_col]
                vals.append(float(sub.iloc[0]) if not sub.empty else 0.0)
            bars = ax.bar(x + (j - 1) * w, vals, w, label=cond, color=COLORS_COND[cond],
                          alpha=0.85, edgecolor='black', linewidth=0.5)
            for bar, v in zip(bars, vals):
                ax.annotate(f'{v:.2f}', xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                            xytext=(0, 2), textcoords='offset points', ha='center', fontsize=7)
        ax.set_xticks(x)
        ax.set_xticklabels([a.split()[0] for a in agents], fontsize=9, rotation=15, ha='right')
        ax.set_ylabel(value_col)
        ax.set_title(f'{ds}\n{bl}', fontweight='bold', fontsize=10)
        ax.set_ylim(*ylim)
        ax.grid(True, alpha=0.3, axis='y')
        ax.legend(fontsize=8)
    fig.suptitle(title, fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_PATH, out_name), dpi=150, bbox_inches='tight')
    plt.show()
    print(f"  wrote {out_name}")


if not met.empty:
    _plot_grouped_bar(met, 'DA',
                      'Module 4  Decision Accuracy per condition and config',
                      'module4v3_accuracy_by_condition.png')

# Calibration curves per (Dataset, BRAS_label)
if not vdf.empty:
    groups = list(vdf.groupby(['Dataset', 'BRAS_label']))
    n = len(groups)
    fig, axes = plt.subplots(1, n, figsize=(7 * n, 7), squeeze=False)
    for idx, ((ds, bl), dd) in enumerate(groups):
        ax = axes[0, idx]
        for cond in ['C1', 'C2', 'C3']:
            cd = dd[dd['Condition'] == cond]
            if cd.empty:
                continue
            edges = np.linspace(0, 1, N_CAL_BINS + 1)
            xs, ys = [], []
            for b in range(N_CAL_BINS):
                mask = (cd['Confidence'] > edges[b]) & (cd['Confidence'] <= edges[b + 1])
                if mask.sum() >= 2:
                    xs.append(float(cd.loc[mask, 'Confidence'].mean()))
                    ys.append(float(cd.loc[mask, 'Correct'].mean()))
            if xs:
                ax.plot(xs, ys, 'o-', color=COLORS_COND[cond], label=cond,
                        linewidth=2, markersize=6)
        ax.plot([0, 1], [0, 1], '--', color='gray', alpha=0.5)
        ax.set_xlabel('Confidence'); ax.set_ylabel('Accuracy')
        ax.set_title(f'{ds}  {bl}', fontweight='bold', fontsize=10)
        ax.set_xlim(0, 1); ax.set_ylim(0, 1)
        ax.grid(True, alpha=0.3); ax.legend()
    fig.suptitle('Module 4  Calibration curves', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_PATH, 'module4v3_calibration_curves.png'),
                dpi=150, bbox_inches='tight')
    plt.show()
    print("  wrote module4v3_calibration_curves.png")

# EU bar plot (C2, C3 only)
eu_df = met[met['EU'].notna() & met['Condition'].isin(['C2', 'C3'])]
if not eu_df.empty:
    groups = list(eu_df.groupby(['Dataset', 'BRAS_label']))
    n = len(groups)
    fig, axes = plt.subplots(1, n, figsize=(7 * n, 6), squeeze=False)
    for idx, ((ds, bl), grp) in enumerate(groups):
        ax = axes[0, idx]
        agents = list(dict.fromkeys(grp['Agent']))
        x = np.arange(len(agents)); w = 0.3
        for j, cond in enumerate(['C2', 'C3']):
            vals = []
            for ag in agents:
                sub = grp[(grp['Agent'] == ag) & (grp['Condition'] == cond)]['EU']
                vals.append(float(sub.iloc[0]) if not sub.empty else 0.0)
            ax.bar(x + (j - 0.5) * w, vals, w, label=cond, color=COLORS_COND[cond],
                   alpha=0.85, edgecolor='black', linewidth=0.5)
        ax.set_xticks(x); ax.set_xticklabels([a.split()[0] for a in agents], fontsize=9)
        ax.set_ylabel('Explanation Utilization')
        ax.set_title(f'{ds}  {bl}', fontweight='bold', fontsize=10)
        ax.set_ylim(0, 1.05); ax.grid(True, alpha=0.3, axis='y'); ax.legend(fontsize=8)
    fig.suptitle('Module 4  Explanation Utilization', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_PATH, 'module4v3_utilization_bar.png'),
                dpi=150, bbox_inches='tight')
    plt.show()
    print("  wrote module4v3_utilization_bar.png")

# Kappa heatmaps (per condition)
if len(AGENT_DISPLAY) >= 2 and not vdf.empty:
    groups = list(vdf.groupby(['Dataset', 'BRAS_label']))
    n = len(groups)
    fig, axes = plt.subplots(n, 3, figsize=(15, 4 * n), squeeze=False)
    for di, ((ds, bl), dd) in enumerate(groups):
        for ci, cond in enumerate(['C1', 'C2', 'C3']):
            ax = axes[di, ci]
            cd = dd[dd['Condition'] == cond]
            na = len(AGENT_DISPLAY)
            km = np.full((na, na), np.nan)
            for i, a1 in enumerate(AGENT_DISPLAY):
                km[i, i] = 1.0
                for j, a2 in enumerate(AGENT_DISPLAY):
                    if i < j:
                        d1 = cd[cd['Agent'] == a1].set_index('TX_idx')['Decision']
                        d2 = cd[cd['Agent'] == a2].set_index('TX_idx')['Decision']
                        common = d1.index.intersection(d2.index)
                        if len(common) >= 2:
                            km[i, j] = km[j, i] = cohen_kappa_pair(
                                d1.loc[common], d2.loc[common])
            ax.imshow(km, cmap='YlGn', vmin=0, vmax=1, aspect='auto')
            ax.set_xticks(range(na)); ax.set_xticklabels([a.split()[0] for a in AGENT_DISPLAY],
                                                          fontsize=8, rotation=45, ha='right')
            ax.set_yticks(range(na)); ax.set_yticklabels([a.split()[0] for a in AGENT_DISPLAY], fontsize=8)
            ax.set_title(f'{ds} {bl[:10]}...  {cond}', fontsize=9, fontweight='bold')
            for i in range(na):
                for j in range(na):
                    if not np.isnan(km[i, j]):
                        color = 'white' if km[i, j] > 0.6 else 'black'
                        ax.text(j, i, f'{km[i, j]:.2f}', ha='center', va='center',
                                fontsize=9, color=color)
    fig.suptitle('Inter-Agent Agreement (Cohen kappa)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_PATH, 'module4v3_kappa_heatmap.png'),
                dpi=150, bbox_inches='tight')
    plt.show()
    print("  wrote module4v3_kappa_heatmap.png")


## 9. Save results


In [ ]:
met.to_csv(os.path.join(RESULTS_PATH, 'module4v3_llm_results.csv'), index=False)

summary_rows = []
for ds in met['Dataset'].unique():
    for bl in met[met['Dataset'] == ds]['BRAS_label'].unique():
        for cond in ['C1', 'C2', 'C3']:
            sub = met[(met['Dataset'] == ds) & (met['BRAS_label'] == bl) & (met['Condition'] == cond)]
            if sub.empty:
                continue
            summary_rows.append({
                'Dataset': ds, 'BRAS_label': bl, 'Condition': cond,
                'DA_mean': float(sub['DA'].mean()), 'DA_std': float(sub['DA'].std()),
                'ECE_mean': float(sub['ECE'].mean()),
                'EU_mean': float(sub['EU'].mean()) if sub['EU'].notna().any() else None,
                'Kappa': float(sub['Kappa'].iloc[0]) if not sub.empty else None,
            })
pd.DataFrame(summary_rows).to_csv(
    os.path.join(RESULTS_PATH, 'module4v3_llm_summary.csv'), index=False)
if not delt_df.empty:
    delt_df.to_csv(os.path.join(RESULTS_PATH, 'module4v3_llm_deltas.csv'), index=False)

print("""
Files saved:
  module4v3_llm_raw_responses.csv
  module4v3_llm_results.csv
  module4v3_llm_summary.csv
  module4v3_llm_deltas.csv
  module4v3_accuracy_by_condition.png
  module4v3_calibration_curves.png
  module4v3_utilization_bar.png
  module4v3_kappa_heatmap.png

Hypotheses tested:
  H1: DA(C2) > DA(C1)            XAI explanation helps decisions.
  H2: DA(C3) > DA(C2)            Quality scores help further.
  H3: ECE(C3) < ECE(C1)          Richer context improves calibration.
  H4: Kappa(C3) > Kappa(C1)      Agents converge with more info.
  H5: DA(high_BRAS) > DA(low_BRAS) in C2/C3 - XAI quality changes outcomes.
""")
